In [6]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import os
from pathlib import Path

import numpy as np

from dataset.dataset import anonymize_dataset, create_dataset, split_dataset
from dataset.utils import link_dir
from plotting import plot_metrics
from training import *


# Parameters

In [ ]:
n = 3

# Goal
Benchmark the model "yolo11-cls" model on an increasing quantity of data.

The dataset has been already tested and the model too (KFold cross validation).

Just change the training dataset quantity but keep the same test dataset
- 20% -> 40% -> 60% -> 80% -> 100

### Create datasets

In [ ]:
quantities = [0.2, 0.4, 0.6, 0.8, 1.0]
path = "dataset/benchmark"
path_csv = "dataset/benchmark"

anonymize_dataset("dataset/source/csv/sources_kfold_train.csv")

for q in quantities:
    for i in range(n):
        create_dataset("dataset/source/csv/sources_kfold_train.csv", 
                       f"dataset/benchmark/kfold_{q}/{i}/train", 
                       f"dataset/source/csv/benchmark/kfold_{q}/{i}.csv", quantity=q, random=True)
        link_dir("dataset/main/test",
                 f"dataset/benchmark/kfold_{q}/{i}/test")


### Training

In [ ]:
def benchmark_dataset_yolo(config, datasets: list[str], names, n=3):
    # mean of (score, accuracy, recall, f1)
    histories = []
    for ds_path in datasets:
        
        results = []
        for i in range(n):
            name = os.path.basename(ds_path)
            
            ds = Path(ds_path, str(i))
            
            model = train(name, ds, config)
            _, metric = test(model, ds)
            
            results.append(metric[0])

        results = np.asarray(results)
        histories.append([(results.mean(), results.std())])
        
    print(histories)
    plot_metrics(histories, names, metrics=[("accuracy", "number of images")])

In [ ]:
config = {"model": "models/yolo11x-cls.pt", 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512,
        }
# "dataset/benchmark/kfold_0.4/", "dataset/benchmark/kfold_0.6/", "dataset/benchmark/kfold_0.8/", "dataset/main/"
# "40", "60", "80", "100"

# benchmark_dataset_yolo(config, ["dataset/benchmark/kfold_0.2/"], ["20"])
benchmark_dataset_yolo(config, ["dataset/benchmark/kfold_0.2/", 
                                "dataset/benchmark/kfold_0.4/", 
                                "dataset/benchmark/kfold_0.6/", 
                                "dataset/benchmark/kfold_0.8/", 
                                "dataset/benchmark/kfold_1.0/"], 
                       ["20", "40", "60", "80", "100"])

## Benchmark Learning Curve: LogScale

In [3]:
def logscale(start, end, num=10):
    values = np.geomspace(start, end, num=num)
    return np.round(values) * 2

In [4]:
train_set = "dataset/source/csv/sources_kfold_train.csv"
L = logscale(1, 40, 10)

#### Dataset Generation

In [10]:
[df_train, df_val] = split_dataset(train_set, [40, 10], random=True)
df_val

,SOURCE,PATIENT,TYPE,SIDE,PATHO,NAME
0,"""Y:\260211\260211_NOX_L\260211_NOX_L_HD\png\mo...",NOX,retine,L,MER + Rétinal detachment,0022.png
1,"""Y:\260305\260305_AGA_L\260305_AGA_L_HD\png\mo...",AGA,retine,L,glaucoma,0044.png
2,"""Y:\260305\260305_SEG_L\260305_SEG_L_HD\png\mo...",SEG,retine,L,NaN,0060.png
3,"""Y:\260305\260305_VIC0_L\260305_VIC0_L_HD\png\...",VIC,retine,L,NaN,0064.png
4,"""Z:\260430\260430_SIM2264975_L\260430_SIM22649...",SIM,retine,L,HTA (hypertension arterial),0024.png
5,"""Z:\260617\260617_APM_L\260617_APM_L_HD\png\mo...",APM,retine,L,NaN,0012.png
6,"""Y:\260114\260114_ALA_L_1\260114_ALA_L_1_HD\pn...",ALA,retine,L,NaN,0026.png
7,"""Z:\260617\260617_ESM_L\260617_ESM_L_HD\png\mo...",ESM,retine,L,NaN,0014.png
8,"""Z:\260325\260325_ZEN1418552_L\260325_ZEN14185...",ZEN,retine,L,NaN,0002.png
9,"""Z:\260512\260512_MAH1352649_L_2\260512_MAH135...",MAH,retine,L,NaN,0068.png


In [ ]:
# split
[df_train, df_val] = split_dataset(train_set, [80, 20], random=True)

output_dir = "dataset/benchmark/learning_curve"
output_csv = "dataset/source/csv/benchmark/learning_curve"

create_dataset(df_val, f"{output_dir}/val", f"{output_csv}/val.csv")

for n_subject in L:
    size = n_subject * 2
    create_dataset(df_train, f"{output_dir}/{size}/train", f"{output_csv}/{size}.csv", 
                   size, is_random=True, is_subject_level=True)
    link_dir(f"{output_dir}/val",
            f"{output_dir}/{size}/val")